In [12]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import time
import os

print("=" * 70)
print("EXTERNAL DATASET TESTING")
print("=" * 70)

PROJECT_DIR = Path(
    "/srv/data/datasets/Network-Intrusion-Detection-System/"
    "Intrusion_Detection_System"
)

DATASET_DIR = PROJECT_DIR / "dataset" / "cleaned"
MODELS_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"

RESULTS_DIR.mkdir(exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATASET_DIR:", DATASET_DIR)
print("MODELS_DIR :", MODELS_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

EXTERNAL DATASET TESTING
PROJECT_DIR: /srv/data/datasets/Network-Intrusion-Detection-System/Intrusion_Detection_System
DATASET_DIR: /srv/data/datasets/Network-Intrusion-Detection-System/Intrusion_Detection_System/dataset/cleaned
MODELS_DIR : /srv/data/datasets/Network-Intrusion-Detection-System/Intrusion_Detection_System/models
RESULTS_DIR: /srv/data/datasets/Network-Intrusion-Detection-System/Intrusion_Detection_System/results


In [14]:
print("=" * 70)
print("SEARCHING FOR EXTERNAL CSV FILES")
print("=" * 70)

search_locations = [
    Path("/srv/data"),
    Path("/mnt/data"),
]

csv_files = []

for location in search_locations:
    if location.exists():
        for file in location.rglob("*.csv"):
            if (
                "sql" in file.name.lower()
                or "parallel" in file.name.lower()
                or "flow" in file.name.lower()
            ):
                csv_files.append(file)

for file in csv_files:
    print(file)

SEARCHING FOR EXTERNAL CSV FILES


In [15]:
from pathlib import Path

EXTERNAL_DIR = Path(
    "/srv/data/datasets/Network-Intrusion-Detection-System/"
    "Intrusion_Detection_System/dataset/external_test"
)

print("=" * 70)
print("EXTERNAL DATASET FILES")
print("=" * 70)

for file in EXTERNAL_DIR.glob("*"):
    print(file)

EXTERNAL DATASET FILES
/srv/data/datasets/Network-Intrusion-Detection-System/Intrusion_Detection_System/dataset/external_test/sql_newnew_parallel.pcap_Flow.csv
/srv/data/datasets/Network-Intrusion-Detection-System/Intrusion_Detection_System/dataset/external_test/.ipynb_checkpoints


In [16]:
EXTERNAL_CSV_PATH = (
    EXTERNAL_DIR / "sql_newnew_parallel.pcap_Flow.csv"
)

print("File exists:", EXTERNAL_CSV_PATH.exists())

df_external = pd.read_csv(EXTERNAL_CSV_PATH)

print("\nShape:", df_external.shape)

print("\nFirst 10 columns:")
print(df_external.columns.tolist()[:10])

print("\nLast 10 columns:")
print(df_external.columns.tolist()[-10:])

print("\nLabel distribution:")

label_cols = [
    col for col in df_external.columns
    if col.strip().lower() == "label"
]

if label_cols:
    LABEL_COLUMN = label_cols[0]
    print(df_external[LABEL_COLUMN].value_counts())
else:
    print("No Label column found")

File exists: True

Shape: (11, 84)

First 10 columns:
['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Total Fwd Packet', 'Total Bwd packets']

Last 10 columns:
['Fwd Seg Size Min', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min', 'Label']

Label distribution:
Label
SqlInjWeb    10
normal        1
Name: count, dtype: int64


In [17]:
print("=" * 70)
print("LOADING OUR 30 SELECTED FEATURES")
print("=" * 70)

selected_features_path = (
    DATASET_DIR / "selected_features_clean.csv"
)

print("Path:", selected_features_path)
print("File exists:", selected_features_path.exists())

selected_features_df = pd.read_csv(selected_features_path)

print("\nSelected features file shape:")
print(selected_features_df.shape)

print("\nContents:")
print(selected_features_df.head(35))

LOADING OUR 30 SELECTED FEATURES
Path: /srv/data/datasets/Network-Intrusion-Detection-System/Intrusion_Detection_System/dataset/cleaned/selected_features_clean.csv
File exists: True

Selected features file shape:
(1989921, 31)

Contents:
      Fwd IAT Std  Bwd IAT Min  Flow IAT Min  Bwd Packet Length Std  \
0        0.000000            0             3                    0.0   
1        0.000000            0           109                    0.0   
2        0.000000            0            52                    0.0   
3        0.000000            0            34                    0.0   
4        0.000000            0          1022                    0.0   
5        0.000000            0             4                    0.0   
6        0.000000            0            42                    0.0   
7        0.000000            0             4                    0.0   
8        0.000000            0             3                    0.0   
9        0.000000            0             1        

In [18]:
print("=" * 70)
print("EXTERNAL DATASET FEATURE COMPATIBILITY")
print("=" * 70)

# Our model's 30 features in exact training order
selected_features = [
    col.strip()
    for col in selected_features_df.columns
    if col.strip() != "Label"
]

# Clean external column names
external_columns = [
    col.strip()
    for col in df_external.columns
]

# Find matches and missing features
matched_features = [
    feature
    for feature in selected_features
    if feature in external_columns
]

missing_features = [
    feature
    for feature in selected_features
    if feature not in external_columns
]

extra_features = [
    feature
    for feature in external_columns
    if feature not in selected_features
    and feature != LABEL_COLUMN
]

print(f"\nOur required features : {len(selected_features)}")
print(f"External CSV features : {len(external_columns) - 1}")
print(f"Matched features      : {len(matched_features)}")
print(f"Missing features      : {len(missing_features)}")

print("\n" + "=" * 70)
print("MATCHED FEATURES")
print("=" * 70)

for feature in matched_features:
    print("✓", feature)

print("\n" + "=" * 70)
print("MISSING REQUIRED FEATURES")
print("=" * 70)

for feature in missing_features:
    print("✗", feature)

print("\n" + "=" * 70)
print("EXTRA EXTERNAL FEATURES")
print("=" * 70)

for feature in extra_features:
    print("+", feature)

EXTERNAL DATASET FEATURE COMPATIBILITY

Our required features : 30
External CSV features : 83
Matched features      : 27
Missing features      : 3

MATCHED FEATURES
✓ Fwd IAT Std
✓ Bwd IAT Min
✓ Flow IAT Min
✓ Bwd Packet Length Std
✓ Bwd Packet Length Mean
✓ Idle Min
✓ Bwd Packet Length Max
✓ Idle Mean
✓ Packet Length Std
✓ Idle Max
✓ Flow IAT Max
✓ Fwd IAT Max
✓ Packet Length Variance
✓ Average Packet Size
✓ Packet Length Mean
✓ Active Min
✓ FIN Flag Count
✓ Active Std
✓ Flow IAT Std
✓ PSH Flag Count
✓ Active Mean
✓ Fwd IAT Total
✓ ACK Flag Count
✓ Flow Duration
✓ Bwd IAT Std
✓ Subflow Fwd Bytes
✓ Flow IAT Mean

MISSING REQUIRED FEATURES
✗ Avg Bwd Segment Size
✗ Max Packet Length
✗ Min Packet Length

EXTRA EXTERNAL FEATURES
+ Flow ID
+ Src IP
+ Src Port
+ Dst IP
+ Dst Port
+ Protocol
+ Timestamp
+ Total Fwd Packet
+ Total Bwd packets
+ Total Length of Fwd Packet
+ Total Length of Bwd Packet
+ Fwd Packet Length Max
+ Fwd Packet Length Min
+ Fwd Packet Length Mean
+ Fwd Packet Length St

In [19]:
print("=" * 70)
print("ALIGNING EXTERNAL DATASET FEATURES")
print("=" * 70)

# Make a copy
X_external = df_external.copy()

# Clean whitespace from column names
X_external.columns = X_external.columns.str.strip()

# Feature-name mapping
feature_mapping = {
    "Bwd Segment Size Avg": "Avg Bwd Segment Size",
    "Packet Length Max": "Max Packet Length",
    "Packet Length Min": "Min Packet Length"
}

# Show mapping
print("\nFEATURE NAME MAPPING:")
for old_name, new_name in feature_mapping.items():
    print(f"{old_name}  →  {new_name}")

# Rename equivalent features
X_external = X_external.rename(columns=feature_mapping)

# Remove label column
X_external_features = X_external.drop(
    columns=[LABEL_COLUMN],
    errors="ignore"
)

# Extract exactly the 30 features in training order
X_external_aligned = X_external_features[selected_features].copy()

print("\n" + "=" * 70)
print("FINAL FEATURE ALIGNMENT")
print("=" * 70)

print("Required features :", len(selected_features))
print("Aligned features  :", X_external_aligned.shape[1])
print("External samples  :", X_external_aligned.shape[0])

# Final safety check
assert list(X_external_aligned.columns) == selected_features

print("\n✓ All 30 features successfully aligned")
print("✓ Feature order matches training data")

print("\nAligned feature names:")
for i, feature in enumerate(X_external_aligned.columns, 1):
    print(f"{i:2d}. {feature}")

display(X_external_aligned.head())

ALIGNING EXTERNAL DATASET FEATURES

FEATURE NAME MAPPING:
Bwd Segment Size Avg  →  Avg Bwd Segment Size
Packet Length Max  →  Max Packet Length
Packet Length Min  →  Min Packet Length

FINAL FEATURE ALIGNMENT
Required features : 30
Aligned features  : 30
External samples  : 11

✓ All 30 features successfully aligned
✓ Feature order matches training data

Aligned feature names:
 1. Fwd IAT Std
 2. Bwd IAT Min
 3. Flow IAT Min
 4. Bwd Packet Length Std
 5. Bwd Packet Length Mean
 6. Avg Bwd Segment Size
 7. Idle Min
 8. Bwd Packet Length Max
 9. Idle Mean
10. Packet Length Std
11. Idle Max
12. Flow IAT Max
13. Max Packet Length
14. Fwd IAT Max
15. Packet Length Variance
16. Average Packet Size
17. Packet Length Mean
18. Active Min
19. FIN Flag Count
20. Active Std
21. Flow IAT Std
22. PSH Flag Count
23. Active Mean
24. Fwd IAT Total
25. ACK Flag Count
26. Flow Duration
27. Bwd IAT Std
28. Subflow Fwd Bytes
29. Flow IAT Mean
30. Min Packet Length


,Fwd IAT Std,Bwd IAT Min,Flow IAT Min,Bwd Packet Length Std,Bwd Packet Length Mean,Avg Bwd Segment Size,Idle Min,Bwd Packet Length Max,Idle Mean,Packet Length Std,...,Flow IAT Std,PSH Flag Count,Active Mean,Fwd IAT Total,ACK Flag Count,Flow Duration,Bwd IAT Std,Subflow Fwd Bytes,Flow IAT Mean,Min Packet Length
0,0.0,1026,1026,0.0,0.0,0.0,1.630000e+15,0,1.630000e+15,0.0,...,5094.704358,0,0.0,0.0,3,9257,0.0,0,4628.5,0
1,0.0,1374,1374,0.0,0.0,0.0,1.630000e+15,0,1.630000e+15,0.0,...,7172.891188,0,0.0,0.0,3,12892,0.0,0,6446.0,0
2,0.0,0,3317,0.0,0.0,0.0,1.630000e+15,0,1.630000e+15,0.0,...,0.000000,0,0.0,0.0,2,3317,0.0,0,3317.0,0
3,0.0,0,9942,0.0,0.0,0.0,1.630000e+15,0,1.630000e+15,0.0,...,0.000000,0,0.0,0.0,2,9942,0.0,0,9942.0,0
4,0.0,0,10921,0.0,0.0,0.0,1.630000e+15,0,1.630000e+15,0.0,...,0.000000,0,0.0,0.0,2,10921,0.0,0,10921.0,0


In [20]:
print("=" * 70)
print("CLEANING EXTERNAL FEATURES")
print("=" * 70)

# Convert infinity to NaN
X_external_aligned = X_external_aligned.replace(
    [np.inf, -np.inf],
    np.nan
)

print("Missing values before cleaning:")
print(X_external_aligned.isnull().sum().sum())

# Fill missing values using zero
# This should be consistent with the earlier project's
# CICFlowMeter preprocessing if zero was used there.
X_external_aligned = X_external_aligned.fillna(0)

print("Missing values after cleaning:")
print(X_external_aligned.isnull().sum().sum())

print("\nData types:")
print(X_external_aligned.dtypes.value_counts())

print("\n✓ External dataset ready for model prediction")

CLEANING EXTERNAL FEATURES
Missing values before cleaning:
0
Missing values after cleaning:
0

Data types:
float64    20
int64      10
Name: count, dtype: int64

✓ External dataset ready for model prediction


In [22]:
print("=" * 70)
print("LOADING TARGETED-SMOTE STACKING ENSEMBLE")
print("=" * 70)

# Model paths
rf_path = MODELS_DIR / "random_forest_targeted_smote.pkl"
et_path = MODELS_DIR / "extra_trees_targeted_smote.pkl"
xgb_path = MODELS_DIR / "xgboost_targeted_smote.pkl"
meta_path = MODELS_DIR / "stacking_meta_learner_targeted_smote.pkl"

print("\nChecking model files:")

for name, path in {
    "Random Forest": rf_path,
    "Extra Trees": et_path,
    "XGBoost": xgb_path,
    "Meta Learner": meta_path
}.items():
    print(f"{name:15s}: {'✓ EXISTS' if path.exists() else '✗ MISSING'}")

# Stop immediately if any model is missing
required_paths = [rf_path, et_path, xgb_path, meta_path]

if not all(path.exists() for path in required_paths):
    raise FileNotFoundError(
        "One or more Targeted-SMOTE stacking models are missing."
    )

print("\nLoading models...")

rf_targeted = joblib.load(rf_path)
print("✓ Random Forest loaded")

et_targeted = joblib.load(et_path)
print("✓ Extra Trees loaded")

xgb_targeted = joblib.load(xgb_path)
print("✓ XGBoost loaded")

meta_learner_targeted = joblib.load(meta_path)
print("✓ Meta Learner loaded")

print("\n" + "=" * 70)
print("ALL TARGETED-SMOTE STACKING COMPONENTS LOADED")
print("=" * 70)

LOADING TARGETED-SMOTE STACKING ENSEMBLE

Checking model files:
Random Forest  : ✓ EXISTS
Extra Trees    : ✓ EXISTS
XGBoost        : ✓ EXISTS
Meta Learner   : ✓ EXISTS

Loading models...
✓ Random Forest loaded
✓ Extra Trees loaded
✓ XGBoost loaded
✓ Meta Learner loaded

ALL TARGETED-SMOTE STACKING COMPONENTS LOADED
